## Учёт заказов в кофейне

В небольшой кофейне сотрудники принимают заказы от клиентов.
Нужно сделать консольную программу для учёта заказов, их статусов и оплаты.

### Что должна уметь программа

Меню:
```txt
1. Добавить заказ
2. Показать все заказы
3. Изменить статус заказа
4. Показать активные заказы
5. Найти заказы по имени клиента
6. Отметить заказ как оплаченный
7. Показать неоплаченные заказы
8. Показать общую выручку
0. Выход
```

### Данные одного заказа
```txt
Номер заказа
Имя клиента
Состав заказа
Стоимость
Статус
Оплачен / не оплачен
Приоритет
```

### Статусы заказа
```txt
новый
готовится
готов
выдан
отменён
```

### Приоритет заказа
```txt
обычный
срочный
```

### Пример заказа
```txt
№1
Клиент: Анна
Заказ: Капучино и круассан
Стоимость: 450 руб.
Статус: новый
Оплата: не оплачен
Приоритет: обычный
```

### Обязательные условия

1. Номер заказа создаётся автоматически.
2. При добавлении заказа статус всегда новый.
3. При добавлении заказ всегда считается не оплачен.
4. Стоимость заказа должна быть больше 0.
5. Можно изменить статус только на:
   - новый
   - готовится
   - готов
   - выдан
   - отменён
6. Заказ нельзя сделать выданным, если он не оплачен.
7. Отменённый заказ нельзя изменить на другой статус.
8. Активные заказы — это заказы со статусом новый, готовится или готов.
9. Общая выручка считается только по оплаченным заказам, которые не были отменены.
10. При поиске по имени клиента программа должна показывать все заказы этого клиента.
11. Срочные заказы при выводе активных заказов должны показываться первыми.


### Требования к ООП и SOLID

Решение должно быть реализовано в объектно-ориентированном стиле с соблюдением принципов SOLID:

1. Отдельный класс для заказа.
2. Отдельный сервис для управления заказами.
3. Отдельный класс для работы с консольным меню.
4. Отдельная логика проверки статусов и оплаты.
5. Классы не должны брать на себя лишние обязанности.
6. Зависимости между частями программы должны быть минимальными и понятными.

In [ ]:
class Order:
    def __init__(self, order_id, client_name, items, cost, priority = "basic"):
        self.order_id = order_id
        self.client_name = client_name
        self.items = items
        self.cost = cost
        self.status = "new"
        self.paid = False
        self.priority = priority


class OrderService:
    def __init__(self):
        self.orders = []
        self.next_id = 1

    def add_order(self, client_name, items, cost, priority = "basic"):
        order = Order(self.next_id, client_name, items, cost, priority)
        self.orders.append(order)
        self.next_id += 1
        return order
    
    def get_all_orders(self):
        return self.orders
    
    def change_status(self, order_id, new_status):
        order = self._find_order(order_id)
        
        if not order:
            return False, "Заказ не найден"
        
        if order.status == "cancelled":
            return False, "Отменённый заказ нельзя изменить"
        
        if new_status not in ["new", "preparing", "ready", "issued", "cancelled"]:
            return False, "Недопустимый статус"
        
        if new_status == "issued" and not order.paid:
            return False, "Заказ нельзя выдать, если он не оплачен"
        
        order.status = new_status
        return True, "Статус изменён"
    
    def active_orders(self):
        active = [o for o in self.orders if o.status in ["new", "preparing", "ready"]]
        return sorted(active, key=lambda x: x.priority != "express")
    
    def find_orders_by_client(self, client_name):
        return [o for o in self.tickets if o.employee_name.lower() == client_name.lower()]
    
    def _find_order(self, order_id):
        for o in self.orders:
            if o.order_id == order_id:
                return o
        return None
    
    def mark_as_paid(self, order_id):
        order = self._find_order(order_id)

        if not order:
            return False, "Заказ не найден"
        
        order.paid = True
        return True, "Заказ отмечен как оплаченный"
    
    def unpaid_orders(self):
        return [o for o in self.orders if not o.paid]
    
    def revenue(self):
        return sum(o.cost for o in self.orders if o.paid and o.status != "cancelled")
    

class Console:
    def __init__(self, service):
        self.service = service

    def run(self):
        while True:
            print("\n1. Добавить заказ")
            print("2. Показать все заказы")
            print("3. Изменить статус заказа")
            print("4. Показать активные заказы")
            print("5. Найти заказы по имени клиента")
            print("6. Отметить заказ как оплаченный")
            print("7. Показать неоплаченные заказы")
            print("8. Показать общую выручку")
            print("0. Выход")

            choice = input("Выберите действие: ")

            if choice == "1":
                self._add_order()
            elif choice == "2":
                self._show_all_orders()
            elif choice == "3":
                self._change_status()
            elif choice == "4":
                self._show_active_orders()
            elif choice == "5":
                self._find_by_client()
            elif choice == "6":
                self._mark_paid()
            elif choice == "7":
                self._show_unpaid_orders()
            elif choice == "8":
                self._show_revenue()
            elif choice == "0":
                break
            else:
                print("Неверное действие")

    def _add_order(self):
        client = input("Имя клиента: ")
        items = input("Состав заказа: ")
        cost = float(input("Стоимость: "))
        priority = input("Приоритет (basic/express) [basic]: ") or "basic"

        if cost <= 0:
            print("Стоимость должна быть больше 0")
            return
        
        self.service.add.order(client, items, cost, priority)
        print("Заказ добавлен")

    def _show_all_orders(self):
        for o in self.service.get_all_orders():
            print(o)

    def _change_status(self):
        order_id = int(input("Номер заказа: "))
        new_status = input("Новый статус: ")
        success, msg = self.service.change_status(order_id, new_status)
        print(msg)

    def _show_active_orders(self):
        for o in self.service.active_orders():
            print(o)

    def _find_by_client(self):
        name = input("Имя клиента: ")
        orders = self.service.find_orders_by_client(name)
        
        if not orders:
            print("Заказы не найдены")
        
        for o in orders:
            print(o)

    def _mark_paid(self):
        order_id = int(input("Номер заказа: "))
        success, msg = self.service